In [ ]:
# Install/update required packages
# !pip install --upgrade pandas numpy matplotlib scikit-learn tqdm torch torchvision pillow

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autodistill-yolov8 0.1.4 requires ultralytics==8.0.81, but you have ultralytics 8.0.196 which is incompatible.
groundingdino-py 0.4.0 requires supervision==0.6.0, but you have supervision 0.8.0 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.



  Using cached numpy-2.3.3-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached matplotlib-3.10.6-cp312-cp312-win_amd64.whl.metadata (11 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.0 MB 11.2 MB/s eta 0:00:01
   --------------------- ------------------ 5.8/11.0 MB 20.7 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 21.4 MB/s  0:00:00
Using cached numpy-2.3.3-cp312-cp312-win_amd64.whl (12.8 MB)
Using cached matplotlib-3.10.6-cp312-cp312-win_amd64.whl (8.1 MB)
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ------------------------- -------------- 5.5/8.7 MB 28.0 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 28.5 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.2.6

    Uninstalling numpy-2.2.6:

   -------------

In [2]:
# train.py
"""
Train a simple drowsiness classifier (Alert vs Drowsy) using a ResNet-18 backbone.
- Reads Roboflow-exported COCO JSON (_annotations.coco.json)
- Extracts user_tags (Alert, Drowsy) from images[*].extra.user_tags
- Splits into train/val sets
- Trains a classification head
- Saves best model + metadata for reuse with webcam demo
"""

import os, json, random, argparse
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# -------------------------------------------------
# Utils: reproducibility
# -------------------------------------------------
def set_seed(seed=42):
    """Ensure reproducible results across runs/devices."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# -------------------------------------------------
# File scanning
# -------------------------------------------------
def build_path_map(root):
    """Recursively map filename -> fullpath under a given root folder.
       Helps when Roboflow puts images into train/valid/test subfolders."""
    path_map = {}
    for r, _, files in os.walk(root):
        for f in files:
            if f.endswith(".json"):  # skip annotation JSONs
                continue
            path_map[f] = os.path.join(r, f)
    return path_map

# -------------------------------------------------
# COCO annotations processing
# -------------------------------------------------
def process_annotations(coco_data, path_map, prefer_alert_if_both=True):
    """
    Parse images[*].extra.user_tags from Roboflow COCO export.
    Decide labels:
      - If both tags: default to Alert
      - Only drowsy: Drowsy
      - Only alert: Alert
      - Else: skip
    Returns: dict {filename -> label}, plus stats
    """
    image_labels = {}
    unlabeled, missing = 0, 0
    for img in coco_data.get("images", []):
        fname = img["file_name"]
        tags = []
        # Roboflow-style: tags under extra.user_tags
        if isinstance(img.get("extra"), dict):
            tags = img["extra"].get("user_tags", []) or []
        if not tags:
            tags = img.get("user_tags", []) or []
        tags = [t.lower() for t in tags]

        # classification logic
        label = None
        if "drowsy" in tags and "alert" in tags:
            label = "Alert" if prefer_alert_if_both else "Drowsy"
        elif "drowsy" in tags:
            label = "Drowsy"
        elif "alert" in tags:
            label = "Alert"

        if label is None:
            unlabeled += 1
            continue

        if fname in path_map:
            image_labels[fname] = label
        else:
            missing += 1

    return image_labels, {"unlabeled": unlabeled, "missing": missing}

# -------------------------------------------------
# Dataset class
# -------------------------------------------------
class DrowsinessDataset(Dataset):
    """Wraps image paths + labels into a PyTorch Dataset."""
    def __init__(self, image_labels, path_map, transform=None, class_to_idx=None):
        self.keys = [k for k in image_labels.keys() if k in path_map]
        self.paths = [path_map[k] for k in self.keys]
        self.class_to_idx = class_to_idx or {"Drowsy": 0, "Alert": 1}
        self.labels = [self.class_to_idx[image_labels[k]] for k in self.keys]
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        y = self.labels[idx]
        try:
            img = Image.open(p).convert("RGB")
        except (UnidentifiedImageError, OSError) as e:
            raise RuntimeError(f"Failed to open {p}") from e
        if self.transform: img = self.transform(img)
        return img, y

# -------------------------------------------------
# Model creation (ResNet18 backbone + new head)
# -------------------------------------------------
def create_model(unfreeze_last_block=False):
    """
    Load ResNet18 pretrained on ImageNet.
    Freeze backbone (optionally unfreeze last block).
    Replace final FC with small head for 2 classes.
    """
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # freeze all params
    for p in m.parameters():
        p.requires_grad = False
    if unfreeze_last_block:
        for p in m.layer4.parameters():
            p.requires_grad = True
    in_feats = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(in_feats, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(256, 2)  # 0=Drowsy, 1=Alert
    )
    return m

# -------------------------------------------------
# Evaluation helper
# -------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        pred = out.argmax(1)
        total += y.size(0)
        correct += (pred == y).sum().item()
        loss_sum += loss.item() * x.size(0)
    return loss_sum/total, correct/total

# -------------------------------------------------
# Training function
# -------------------------------------------------
def train(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    os.makedirs(args.output_dir, exist_ok=True)

    # Load COCO JSON
    coco_path = os.path.join(args.data_dir, "_annotations.coco.json")
    with open(coco_path, "r") as f:
        coco = json.load(f)

    # Build file map
    path_map = build_path_map(args.data_dir)

    # Parse labels
    image_labels, stats = process_annotations(coco, path_map)
    print(f"Labeled images: {len(image_labels)} | skipped={stats['unlabeled']} | missing={stats['missing']}")

    # Train/val split
    img_files = list(image_labels.keys())
    labels_for_split = [image_labels[k] for k in img_files]
    stratify = labels_for_split if len(set(labels_for_split)) > 1 else None
    train_keys, val_keys = train_test_split(
        img_files, test_size=args.val_ratio, random_state=args.seed, stratify=stratify
    )
    train_labels = {k: image_labels[k] for k in train_keys}
    val_labels   = {k: image_labels[k] for k in val_keys}

    # Data augmentation (train) and normalization (val)
    train_tf = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(7),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

    class_to_idx = {"Drowsy": 0, "Alert": 1}
    train_ds = DrowsinessDataset(train_labels, path_map, train_tf, class_to_idx)
    val_ds   = DrowsinessDataset(val_labels,   path_map, val_tf,   class_to_idx)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=args.batch_size, shuffle=False, num_workers=2)

    # Model + optimizer
    model = create_model(unfreeze_last_block=args.unfreeze_last_block).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=args.lr)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_val_acc = 0.0

    # Training loop
    for epoch in range(1, args.epochs+1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total += y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            loss_sum += loss.item() * x.size(0)

        tr_loss = loss_sum/total
        tr_acc  = correct/total
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)

        print(f"Epoch {epoch:02d}: train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} acc={va_acc:.4f}")

        # Save best model
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_path = os.path.join(args.output_dir, "drowsiness_detection_model.pth")
            torch.save(model.state_dict(), best_path)
            print(f"  ✔ Saved best weights to {best_path}")

    print(f"Training complete. Best val acc: {best_val_acc:.4f}")

# -------------------------------------------------
# CLI entrypoint
# -------------------------------------------------
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("../data/labled_images",   type=str, required=True, help="Folder with _annotations.coco.json and images")
    ap.add_argument("../data/CNN", type=str, default="outputs")
    ap.add_argument("10",     type=int, default=10)
    ap.add_argument("16", type=int, default=32)
    ap.add_argument("1e-3",         type=float, default=1e-3)
    ap.add_argument("0.2",  type=float, default=0.2)
    ap.add_argument("42",       type=int, default=42)
    ap.add_argument("unfreeze_last_block = True")
    args = ap.parse_args()
    train(args)


TypeError: 'required' is an invalid argument for positionals

In [ ]:
MODEL_PATH = "outputs/drowsiness_detection_model.pth"